# ⚡ GravitySort — GPU-Accelerated Sorting & ML Reduction Framework
**Kaggle GPU Notebook — Full Build & Benchmark**

This notebook builds and runs the complete GravitySort CUDA C++ project on Kaggle's T4/P100 GPU.

| Section | What it does |
|---|---|
| 0 | GPU info & CUDA version check |
| 1 | Clone repo & verify structure |
| 2 | CMake build |
| 3 | Sorting kernels (Bitonic / Radix / Odd-Even) |
| 4 | ML Reduction kernels |
| 5 | Memory demos (bank conflicts + streams) |
| 6 | Tensor operations |
| 7 | Google Benchmark |
| 8 | Nsight Compute profiling |
| 9 | Roofline model plot |
| 10 | Python visualization |


## Section 0 — GPU Environment Check

In [ ]:
import subprocess, os, sys

def run(cmd, **kw):
    """Run shell command and print output."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr, file=sys.stderr)
    return result

print('=' * 60)
print('GPU Info')
print('=' * 60)
run('nvidia-smi')
run('nvcc --version')
run('cmake --version')
run('nproc')  # CPU cores for parallel build

## Section 1 — Upload / Clone Project

In [ ]:
import os

# Repo: https://github.com/ThronAxis/GravitySort
GITHUB_USER = 'ThronAxis'
GITHUB_REPO = 'GravitySort'
ZIP_URL     = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO}/archive/refs/heads/main.zip'
ZIP_PATH    = f'/kaggle/working/{GITHUB_REPO}.zip'
EXTRACT_DIR = f'/kaggle/working/{GITHUB_REPO}-main'
PROJECT_DIR = EXTRACT_DIR   # GravitySort IS the repo root now

if os.path.isdir(PROJECT_DIR):
    print(f'Already exists: {PROJECT_DIR}')
else:
    print(f'Downloading from GitHub...')
    r = run(f'wget -q --show-progress -O {ZIP_PATH} "{ZIP_URL}"')
    if r.returncode != 0:
        r = run(f'curl -L -o {ZIP_PATH} "{ZIP_URL}"')
    assert r.returncode == 0, 'Download FAILED'
    print('Extracting...')
    run(f'unzip -q {ZIP_PATH} -d /kaggle/working/')

assert os.path.isdir(PROJECT_DIR), f'Project not found at {PROJECT_DIR}'
print(f'Project root: {PROJECT_DIR}')
run(f'find {PROJECT_DIR} -type f | sort')


## Section 1b — Install Dependencies

In [ ]:
import shutil, glob

print('=' * 55)
print('KAGGLE DEPENDENCY AUDIT')
print('=' * 55)

# 1. System tools
need_apt = []
for t in ['nvcc', 'cmake', 'ninja', 'ncu', 'nsys', 'git']:
    p = shutil.which(t)
    print(f'  {"OK  " if p else "MISS"} {t:<8} {p or ""}')
    if not p and t in ('cmake', 'ninja'): need_apt.append(t)

# 2. Install missing build tools
if need_apt:
    print('\nInstalling via apt:', need_apt)
    run('apt-get update -qq')
    run('apt-get install -y -q cmake ninja-build libglfw3-dev libglew-dev')
else:
    run('apt-get install -y -q libglfw3-dev libglew-dev 2>/dev/null || true')

# 3. Python packages
print('\nPython packages:')
for p in ['numpy', 'matplotlib', 'scipy', 'pygame']:
    try:
        m = __import__(p); print(f'  OK   {p:<12} {m.__version__}')
    except ImportError:
        print(f'  installing {p}...'); run(f'pip install {p} -q')

# 4. CUDA libraries
print('\nCUDA libraries:')
for lib in ['libcublas.so', 'libcudart.so', 'libnvToolsExt.so']:
    found = glob.glob(f'/usr/local/cuda/**/{lib}*', recursive=True)
    print(f'  {"OK  " if found else "MISS"} {lib}')

# 5. Thrust headers
thrust_ok = os.path.isdir('/usr/local/cuda/include/thrust')
print(f'  {"OK  " if thrust_ok else "MISS"} Thrust headers (/usr/local/cuda/include/thrust)')

print('\nAll checks done — ready for Section 2 (CMake Build).')


## Section 2 — CMake Build

In [ ]:
BUILD_DIR = '/kaggle/working/build'
os.makedirs(BUILD_DIR, exist_ok=True)

# Detect GPU architecture
import re
smi = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip()
print(f'GPU: {smi}')

# Map GPU name → CUDA arch
arch_map = {'T4': '75', 'V100': '70', 'P100': '60', 'A100': '80', 'A6000': '86'}
arch = next((v for k,v in arch_map.items() if k.lower() in smi.lower()), '75')
print(f'CUDA Architecture: sm_{arch}')

# CMake configure
print('\n── CMake Configure ──')
result = run(
    f'cmake -S {PROJECT_DIR} -B {BUILD_DIR} '
    f'-DCMAKE_CUDA_ARCHITECTURES={arch} '
    f'-DCMAKE_BUILD_TYPE=Release '
    f'-G Ninja',
    cwd=BUILD_DIR
)
assert result.returncode == 0, 'CMake configure failed!'

# Build (parallel)
print('\n── CMake Build (all targets) ──')
ncpu = int(subprocess.run('nproc', shell=True, capture_output=True, text=True).stdout.strip())
result = run(f'cmake --build {BUILD_DIR} --parallel {ncpu}', cwd=BUILD_DIR)
assert result.returncode == 0, 'Build failed!'
print('\n✓ Build complete')

# List built executables
run(f'ls -lh {BUILD_DIR}/*.cu.o {BUILD_DIR}/bitonic_sort {BUILD_DIR}/radix_sort {BUILD_DIR}/odd_even_sort {BUILD_DIR}/reduction_demo 2>/dev/null || ls {BUILD_DIR}')

## Section 3 — Sorting Kernels

In [ ]:
print('=' * 60)
print('BITONIC SORT — Shared-memory tiled + warp shuffle')
print('=' * 60)
for N in [1<<20, 1<<24, 1<<26]:
    print(f'\nN = {N:,}  ({N>>20}M elements)')
    run(f'{BUILD_DIR}/bitonic_sort {N}')

In [ ]:
print('=' * 60)
print('RADIX SORT — 4-pass LSD, stream-pipelined')
print('=' * 60)
for N in [1<<20, 1<<24, 1<<26]:
    print(f'\nN = {N:,}  ({N>>20}M elements)')
    run(f'{BUILD_DIR}/radix_sort {N}')

In [ ]:
print('=' * 60)
print('ODD-EVEN SORT — Coalesced baseline (small N only)')
print('=' * 60)
run(f'{BUILD_DIR}/odd_even_sort 65536')

## Section 4 — ML Reduction Kernels

In [ ]:
print('=' * 60)
print('REDUCTION KERNELS — 4 variants vs Thrust')
print('=' * 60)
for N in [1<<24, 1<<25, 1<<27]:
    print(f'\nN = {N:,}  ({N>>20}M floats)')
    run(f'{BUILD_DIR}/reduction_demo {N}')

## Section 5 — Memory Optimization Demos

In [ ]:
print('=' * 60)
print('SHARED MEMORY — Bank Conflict Analysis')
print('=' * 60)
run(f'{BUILD_DIR}/shared_mem_demo')

In [ ]:
print('=' * 60)
print('STREAM CONCURRENCY — H2D + Kernel + D2H Overlap')
print('=' * 60)
run(f'{BUILD_DIR}/streams_demo')

## Section 6 — GravityTensor Operations

In [ ]:
print('=' * 60)
print('GRAVITY TENSOR — slice / reshape / to_host / to_device')
print('=' * 60)
run(f'{BUILD_DIR}/tensor_demo')

## Section 7 — Google Benchmark

In [ ]:
print('=' * 60)
print('GOOGLE BENCHMARK — Sort variants')
print('=' * 60)
run(f'{BUILD_DIR}/bench_sort --benchmark_format=console --benchmark_repetitions=3')

print('\n' + '=' * 60)
print('GOOGLE BENCHMARK — Reduction variants')
print('=' * 60)
run(f'{BUILD_DIR}/bench_reduce --benchmark_format=console --benchmark_repetitions=3')

## Section 8 — Nsight Compute Profiling
> **Note**: Nsight Compute (`ncu`) requires root or relaxed perf_event_paranoid.  
> On Kaggle, use `--set full` and `--clock-control none` to avoid permission issues.

In [ ]:
NCU_FLAGS = (
    '--set full '
    '--clock-control none '
    '--metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,'
    'l1tex__t_sector_hit_rate.pct,'
    'smsp__sass_average_data_bytes_per_sector_mem_shared_op_ld.pct '
)

print('── Profiling: Bitonic Sort ──')
run(f'ncu {NCU_FLAGS} --target-processes all {BUILD_DIR}/bitonic_sort 4194304')

print('\n── Profiling: Reduction (Vectorized) ──')
run(f'ncu {NCU_FLAGS} --target-processes all {BUILD_DIR}/reduction_demo 4194304')

In [ ]:
print('── Nsight Systems: Stream concurrency timeline ──')
run(f'nsys profile --trace=cuda,nvtx --output=/kaggle/working/streams_profile '
    f'{BUILD_DIR}/streams_demo')
print('Profile saved → /kaggle/working/streams_profile.nsys-rep')
print('Download and open with Nsight Systems GUI for timeline view.')

## Section 9 — Roofline Model

In [ ]:
import subprocess, os, sys
sys.path.insert(0, '/kaggle/working/GravitySort/profiling')

os.makedirs('/kaggle/working/GravitySort/profiling', exist_ok=True)
run(f'python {PROJECT_DIR}/profiling/roofline.py '
    f'--output /kaggle/working/roofline.png')

from IPython.display import Image, display
display(Image('/kaggle/working/roofline.png'))

## Section 10 — Python Visualization (Matplotlib)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# ── Simulate bitonic sort steps on small array ────────────────────────────
N = 64
arr = np.random.permutation(N).astype(float)

def bitonic_steps(a):
    steps = [a.copy()]
    n = len(a)
    k = 2
    while k <= n:
        j = k // 2
        while j >= 1:
            for i in range(n):
                l = i ^ j
                if l > i:
                    asc = (i & k) == 0
                    if (asc and a[i] > a[l]) or (not asc and a[i] < a[l]):
                        a[i], a[l] = a[l], a[i]
            steps.append(a.copy())
            j //= 2
        k *= 2
    return steps

steps = bitonic_steps(arr.copy())
# Sub-sample for fast animation
steps = steps[::max(1, len(steps)//80)]

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
for spine in ax.spines.values(): spine.set_color('#30363d')
ax.tick_params(colors='#8b949e')
ax.set_title(f'GravitySort ⚡ Bitonic Sort Animation  N={N}',
             color='#f0f6fc', fontsize=13, fontweight='bold')

colors = plt.cm.plasma(np.linspace(0.1, 0.95, N))
bars = ax.bar(range(N), steps[0], color=colors, width=0.85, edgecolor='none')
ax.set_ylim(0, N + 2)

info = ax.text(0.02, 0.95, '', transform=ax.transAxes,
               color='#8b949e', fontsize=9, va='top')

def update(frame):
    data = steps[frame]
    sorted_colors = plt.cm.plasma(data / N)
    for bar, h, c in zip(bars, data, sorted_colors):
        bar.set_height(h)
        bar.set_color(c)
    info.set_text(f'Step {frame+1}/{len(steps)}')
    return bars

ani = animation.FuncAnimation(fig, update, frames=len(steps),
                               interval=80, blit=False)
plt.tight_layout()
HTML(ani.to_jshtml())

## Section 11 — Unit Tests


In [ ]:
print('=' * 60)
print('RUNNING ALL UNIT TESTS')
print('=' * 60)
r1 = run(f'{BUILD_DIR}/test_sort   --gtest_color=yes')
r2 = run(f'{BUILD_DIR}/test_reduce --gtest_color=yes')

if r1.returncode == 0 and r2.returncode == 0:
    print('\n✓ ALL TESTS PASSED')
else:
    print('\n✗ SOME TESTS FAILED — review output above')

## Summary — Results Table
Fill in after running above cells:

| Kernel | N | Time (µs) | Bandwidth (GB/s) | vs Thrust |
|---|---|---|---|---|
| Bitonic Sort | 16M | — | — | — |
| Radix Sort | 16M | — | — | — |
| Reduction (naive) | 32M | — | — | — |
| Reduction (vec4) | 32M | — | — | — |

---
*GravitySort — CUDA/Kernel/ML Systems Engineering Portfolio  ·  June 2026*
